# 🎵 Spotify Recommendation System (End-to-End)
Notebook ini berisi tahapan lengkap pembuatan sistem rekomendasi musik berbasis *Content-Based Filtering*, dari proses Load Data hingga Modeling, dibuat serapi mungkin.

## 1. Import Library & Load Data
Mengimpor library yang dibutuhkan dan mengunduh dataset dari Kaggle.

In [ ]:
# Import library dasar
import os
import pandas as pd
import numpy as np

# Import library untuk Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Import kagglehub untuk download data
import kagglehub

# Download dataset
path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")
dataset_path = os.path.join(path, "dataset.csv")

# Membaca data menggunakan Pandas
dataset = pd.read_csv(dataset_path)

# Menampilkan 5 data teratas
dataset.head()

## 2. Data Cleaning
Membersihkan data dari kolom yang tidak penting, data kosong (*missing values*), dan duplikat lagu agar hasil rekomendasi lebih akurat.

In [ ]:
# 1. Menghapus kolom 'Unnamed: 0' karena hanya berupa index tak bernilai
if 'Unnamed: 0' in dataset.columns:
    dataset = dataset.drop(['Unnamed: 0'], axis=1)

# 2. Menghapus baris yang memiliki nilai kosong (NaN)
dataset = dataset.dropna()

# 3. Menghapus lagu duplikat berdasarkan judul dan artis
# Kita urutkan berdasarkan popularity tertinggi agar versi terpopuler yang dipertahankan
dataset = dataset.sort_values('popularity', ascending=False)
dataset = dataset.drop_duplicates(subset=['track_name', 'artists'], keep='first')

# 4. Reset index setelah penghapusan baris
dataset = dataset.reset_index(drop=True)

print(f"Data bersih siap digunakan! Jumlah lagu: {dataset.shape[0]}")

## 3. Feature Engineering
Kita akan membuat sistem rekomendasi berdasarkan *Genre* dan *Artis*. Oleh karena itu, kita perlu menggabungkan kedua teks ini menjadi satu fitur (kolom `tags`).

In [ ]:
# Menggabungkan genre dan artists menjadi satu teks utuh
dataset['tags'] = dataset['track_genre'] + " " + dataset['artists']

# Menampilkan hasil gabungan
dataset[['track_name', 'artists', 'track_genre', 'tags']].head()

## 4. Modeling (TF-IDF & Cosine Similarity)
Model komputer tidak bisa membaca teks. Kita gunakan `TfidfVectorizer` untuk mengubah teks (tags) menjadi matriks angka. 
Lalu kita buat fungsi yang menggunakan `linear_kernel` (Cosine Similarity) untuk menghitung kemiripan lagu yang dicari dengan semua lagu di dataset.

In [ ]:
# 1. Inisialisasi TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')

# 2. Mengubah teks 'tags' menjadi matriks vektor
tfidf_matrix = tfidf.fit_transform(dataset['tags'])
print(f"Bentuk matriks TF-IDF: {tfidf_matrix.shape}")

# 3. Membuat pemetaan judul lagu ke index untuk pencarian cepat
indices = pd.Series(dataset.index, index=dataset['track_name']).drop_duplicates()

# 4. Fungsi Sistem Rekomendasi
def get_recommendations(title, tfidf_matrix=tfidf_matrix, df=dataset, indices=indices):
    # Cek ketersediaan lagu
    if title not in indices:
        return "Lagu tidak ditemukan. Pastikan huruf besar/kecilnya sama persis."
        
    # Ambil index lagu
    idx = indices[title]
    if type(idx) == pd.Series:
        idx = idx.iloc[0] # Ambil yang pertama jika ada judul kembar
        
    # Hitung Cosine Similarity untuk lagu ini vs seluruh lagu lain
    sim_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    
    # Ambil 10 urutan index dengan skor tertinggi (mengabaikan index 0)
    top_indices = sim_scores.argsort()[::-1][1:11]
    
    # Tampilkan hasilnya
    return df[['track_name', 'artists', 'track_genre']].iloc[top_indices]

## 5. Testing Sistem Rekomendasi
Mari kita coba jalankan model yang sudah kita buat!

In [ ]:
judul_lagu = 'Hold On'

print(f"\n🎵 Rekomendasi lagu mirip '{judul_lagu}':")
get_recommendations(judul_lagu).head()